# Chapter 5 — Fourier Series and Orthogonal Expansions

This interactive notebook develops Fourier analysis through **geometry, computation, and visualization**. It accompanies the chapter rather than replacing its proofs.

You will investigate:

- orthogonal projection and best approximation in (L^2);
- real and complex Fourier coefficients;
- pointwise, uniform, and mean-square convergence;
- the Dirichlet and Fejér kernels;
- the Gibbs phenomenon;
- Bessel's inequality and Parseval's identity;
- coefficient decay and smoothness;
- half-range sine series and the heat equation;
- translation, convolution, Euler's sine product, and Wallis' product.

> **How to use the notebook.** Run the cells from top to bottom. Change the controls, press the buttons, predict the result first, and then compare your prediction with the graph or numerical certificate.


## 1. Computational setup

On one period ([-pi,pi]), the real Fourier coefficients are

\[
a_0=\frac1\pi\int_{-\pi}^{\pi}f(x)\,dx,
\qquad
a_n=\frac1\pi\int_{-\pi}^{\pi}f(x)\cos(nx)\,dx,
\qquad
b_n=\frac1\pi\int_{-\pi}^{\pi}f(x)\sin(nx)\,dx.
\]

The (N)-th partial sum is

\[
S_Nf(x)=\frac{a_0}{2}+\sum_{n=1}^{N}
\bigl(a_n\cos(nx)+b_n\sin(nx)\bigr).
\]

The next cell imports the numerical and interactive tools and defines reusable Fourier routines. All integrals are approximated on a fine, equally spaced periodic grid.


In [ ]:
# Core imports used throughout the notebook.
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, HTML, Math, clear_output

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (9, 4.8), "font.size": 11})

PI = np.pi
TWO_PI = 2.0 * np.pi


def periodic_wrap(x):
    """Map real numbers to the representative interval [-pi, pi)."""
    return (np.asarray(x) + PI) % TWO_PI - PI


def l2_norm(values):
    """Approximate the L2 norm on one period using a periodic grid."""
    values = np.asarray(values)
    return np.sqrt(TWO_PI * np.mean(np.abs(values) ** 2))


def real_fourier_coefficients(func, max_mode, samples=16384):
    """Approximate a_0, a_n, and b_n for a 2*pi-periodic function."""
    x = np.linspace(-PI, PI, samples, endpoint=False)
    values = np.asarray(func(x), dtype=float)
    a0 = 2.0 * np.mean(values)
    modes = np.arange(1, max_mode + 1)
    a = np.array([2.0 * np.mean(values * np.cos(n * x)) for n in modes])
    b = np.array([2.0 * np.mean(values * np.sin(n * x)) for n in modes])
    return a0, a, b


def real_fourier_sum(x, a0, a, b, max_mode=None):
    """Evaluate a real Fourier polynomial from its coefficients."""
    if max_mode is None:
        max_mode = min(len(a), len(b))
    x = np.asarray(x)
    result = np.full_like(x, a0 / 2.0, dtype=float)
    for n in range(1, max_mode + 1):
        result += a[n - 1] * np.cos(n * x) + b[n - 1] * np.sin(n * x)
    return result


def fejer_sum(x, a0, a, b, order):
    """Evaluate sigma_N f using the triangular Fejer weights."""
    x = np.asarray(x)
    result = np.full_like(x, a0 / 2.0, dtype=float)
    for n in range(1, min(order, len(a), len(b)) + 1):
        weight = 1.0 - n / (order + 1.0)
        result += weight * (a[n - 1] * np.cos(n * x) + b[n - 1] * np.sin(n * x))
    return result


def sign_function(x):
    """Normalized periodic sign function on (-pi, pi)."""
    return np.sign(periodic_wrap(x))


FUNCTIONS = {
    "Sign function": sign_function,
    "Sawtooth  f(x)=x": lambda x: periodic_wrap(x),
    "Absolute value  f(x)=|x|": lambda x: np.abs(periodic_wrap(x)),
    "Square  f(x)=x^2": lambda x: periodic_wrap(x) ** 2,
    "Rectangular pulse": lambda x: (np.abs(periodic_wrap(x)) < PI / 2).astype(float),
    "Smooth  f(x)=exp(cos x)": lambda x: np.exp(np.cos(x)),
}

display(HTML(
    "<div style='padding:10px;border-left:4px solid #2a6fbb;background:#eef6ff'>"
    "<b>Setup complete.</b> Numerical Fourier tools and the function gallery are ready."
    "</div>"
))


## 2. Orthogonal projection is the best mean-square approximation

For the (L^2) inner product

\[
\langle f,g\rangle=\int_{-\pi}^{\pi}f(x)\overline{g(x)}\,dx,
\qquad
\|f\|_2^2=\langle f,f\rangle,
\]

the Fourier partial sum (P_Nf=S_Nf) is the orthogonal projection of (f) onto

\[
V_N=\operatorname{span}\{1,\cos x,\sin x,\ldots,\cos Nx,\sin Nx\}.
\]

Consequently, every (Q\in V_N) satisfies the Pythagorean identity

\[
\|f-Q\|_2^2
=\|f-P_Nf\|_2^2+\|P_Nf-Q\|_2^2,
\]

so (P_Nf) uniquely minimizes the mean-square error. The experiment perturbs the Fourier coefficients and verifies this identity numerically.


In [ ]:
# Interactive verification of the best-approximation theorem.
projection_function = widgets.Dropdown(options=list(FUNCTIONS), value="Absolute value  f(x)=|x|", description="Function:")
projection_modes = widgets.IntSlider(value=5, min=1, max=25, step=1, description="N:")
projection_perturbation = widgets.FloatSlider(value=0.35, min=0.0, max=1.5, step=0.05, description="Perturb:")
projection_button = widgets.Button(description="Compare approximations", button_style="primary", icon="balance-scale")
projection_output = widgets.Output()


def run_projection_experiment(_=None):
    """Compare the Fourier projection with a perturbed polynomial in V_N."""
    with projection_output:
        clear_output(wait=True)
        func = FUNCTIONS[projection_function.value]
        n_modes = projection_modes.value
        x = np.linspace(-PI, PI, 12001, endpoint=False)
        target = func(x)
        a0, a, b = real_fourier_coefficients(func, n_modes)
        projection = real_fourier_sum(x, a0, a, b)

        # Create a reproducible perturbation of every coefficient.
        rng = np.random.default_rng(2026 + n_modes)
        direction = rng.normal(size=1 + 2 * n_modes)
        direction /= np.linalg.norm(direction)
        scale = projection_perturbation.value
        q_a0 = a0 + scale * direction[0]
        q_a = a + scale * direction[1:1 + n_modes]
        q_b = b + scale * direction[1 + n_modes:]
        competitor = real_fourier_sum(x, q_a0, q_a, q_b)

        error_projection_sq = l2_norm(target - projection) ** 2
        error_competitor_sq = l2_norm(target - competitor) ** 2
        distance_sq = l2_norm(projection - competitor) ** 2
        identity_gap = error_competitor_sq - error_projection_sq - distance_sq

        fig, ax = plt.subplots()
        ax.plot(x, target, color="black", linewidth=2, label="target f")
        ax.plot(x, projection, color="#1565c0", label=f"Fourier projection P_{n_modes}f")
        ax.plot(x, competitor, color="#d95f02", alpha=0.85, label="perturbed Q")
        ax.set_title("Orthogonal projection versus another polynomial in the same space")
        ax.legend(loc="best")
        plt.show()

        display(HTML(
            f"<b>Squared errors:</b> projection = {error_projection_sq:.7f}, "
            f"competitor = {error_competitor_sq:.7f}<br>"
            f"<b>Pythagorean residual:</b> {identity_gap:+.3e} "
            "(small numerical quadrature error is expected)."
        ))


projection_button.on_click(run_projection_experiment)
display(widgets.VBox([
    widgets.HBox([projection_function, projection_modes]),
    projection_perturbation,
    projection_button,
    projection_output,
]))
run_projection_experiment()


## 3. Fourier coefficient laboratory: parity and mode content

If (f) is even, then (b_n=0); if (f) is odd, then (a_0=a_n=0). These cancellations follow directly from parity:

\[
\text{odd integrand on }[-\pi,\pi]
\quad\Longrightarrow\quad
\int_{-\pi}^{\pi}h(x)\,dx=0.
\]

The coefficient magnitude

\[
A_n=\sqrt{a_n^2+b_n^2}
\]

measures the strength of frequency (n). Inspect the table and spectrum, and compare the numerical zeros with the expected parity of the selected function.


In [ ]:
# Compute and display real Fourier coefficients and their amplitudes.
coefficient_function = widgets.Dropdown(options=list(FUNCTIONS), value="Sawtooth  f(x)=x", description="Function:")
coefficient_modes = widgets.IntSlider(value=12, min=3, max=40, step=1, description="Modes:")
coefficient_button = widgets.Button(description="Compute coefficients", button_style="info", icon="calculator")
coefficient_output = widgets.Output()


def show_coefficients(_=None):
    """Display a coefficient table and amplitude spectrum."""
    with coefficient_output:
        clear_output(wait=True)
        n_modes = coefficient_modes.value
        func = FUNCTIONS[coefficient_function.value]
        a0, a, b = real_fourier_coefficients(func, n_modes)
        modes = np.arange(1, n_modes + 1)
        amplitudes = np.sqrt(a * a + b * b)

        rows = "".join(
            f"<tr><td>{n}</td><td>{a[n-1]:+.7f}</td><td>{b[n-1]:+.7f}</td>"
            f"<td>{amplitudes[n-1]:.7f}</td></tr>"
            for n in range(1, min(n_modes, 12) + 1)
        )
        display(HTML(
            f"<b>a<sub>0</sub> = {a0:+.7f}</b>"
            "<table style='border-collapse:collapse;margin-top:8px'>"
            "<tr><th style='padding:4px 12px'>n</th><th style='padding:4px 12px'>a_n</th>"
            "<th style='padding:4px 12px'>b_n</th><th style='padding:4px 12px'>A_n</th></tr>"
            + rows + "</table>"
        ))

        fig, ax = plt.subplots()
        ax.stem(modes, amplitudes, basefmt=" ", linefmt="#1565c0", markerfmt="o")
        ax.set_yscale("log")
        ax.set_xlabel("frequency n")
        ax.set_ylabel(r"amplitude $\sqrt{a_n^2+b_n^2}$")
        ax.set_title("Fourier amplitude spectrum")
        plt.show()


coefficient_button.on_click(show_coefficients)
display(widgets.VBox([
    widgets.HBox([coefficient_function, coefficient_modes]),
    coefficient_button,
    coefficient_output,
]))
show_coefficients()


## 4. Build a Fourier partial sum mode by mode

For a piecewise smooth (2pi)-periodic function, the Dirichlet–Jordan theorem gives

\[
S_Nf(x)\longrightarrow \frac{f(x+)+f(x-)}2.
\]

Thus the limit equals (f(x)) at continuity points and the midpoint of the jump at a discontinuity. Use the play control to add frequencies one at a time. Watch how low frequencies capture global shape while higher frequencies resolve finer detail.


In [ ]:
# Animate the construction of a Fourier partial sum.
sum_function = widgets.Dropdown(options=list(FUNCTIONS), value="Rectangular pulse", description="Function:")
sum_modes = widgets.IntSlider(value=8, min=0, max=60, step=1, description="N:")
sum_play = widgets.Play(value=8, min=0, max=60, step=1, interval=180, description="Play")
widgets.jslink((sum_play, "value"), (sum_modes, "value"))
sum_output = widgets.Output()


def draw_partial_sum(change=None):
    """Plot the selected function and its current Fourier partial sum."""
    with sum_output:
        clear_output(wait=True)
        n_modes = sum_modes.value
        func = FUNCTIONS[sum_function.value]
        x = np.linspace(-PI, PI, 4001)
        a0, a, b = real_fourier_coefficients(func, max(1, n_modes))
        approximation = real_fourier_sum(x, a0, a, b, max_mode=n_modes)

        fig, ax = plt.subplots()
        ax.plot(x, func(x), color="black", linewidth=2.2, label="normalized target")
        ax.plot(x, approximation, color="#1565c0", linewidth=1.6, label=fr"$S_{{{n_modes}}}f$")
        ax.set_xlim(-PI, PI)
        ax.set_title("Fourier reconstruction mode by mode")
        ax.legend(loc="best")
        plt.show()


sum_function.observe(draw_partial_sum, names="value")
sum_modes.observe(draw_partial_sum, names="value")
display(widgets.VBox([
    widgets.HBox([sum_function, sum_play, sum_modes]),
    sum_output,
]))
draw_partial_sum()


## 5. Different modes of convergence

Fourier analysis requires us to distinguish:

\[
\begin{aligned}
&\text{pointwise:} && S_Nf(x)\to f(x)\quad\text{for each fixed }x,\\
&\text{uniform:} && \|S_Nf-f\|_\infty\to0,\\
&\text{mean square:} && \|S_Nf-f\|_2\to0.
\end{aligned}
\]

At a jump, uniform convergence is impossible because every (S_Nf) is continuous. Nevertheless, (L^2) convergence still occurs, and uniform convergence can hold on closed sets separated from the jumps. The experiment reports all three diagnostics.


In [ ]:
# Compare L2 error, global maximum error, and maximum error away from jumps.
convergence_function = widgets.Dropdown(
    options=["Sign function", "Sawtooth  f(x)=x", "Absolute value  f(x)=|x|"],
    value="Sign function",
    description="Function:",
)
convergence_modes = widgets.IntSlider(value=25, min=1, max=150, step=1, description="N:")
convergence_button = widgets.Button(description="Measure convergence", button_style="success", icon="check")
convergence_output = widgets.Output()


def convergence_diagnostics(_=None):
    """Measure Fourier errors globally and away from discontinuities."""
    with convergence_output:
        clear_output(wait=True)
        name = convergence_function.value
        func = FUNCTIONS[name]
        n_modes = convergence_modes.value
        x = np.linspace(-PI, PI, 30001, endpoint=False)
        target = func(x)
        a0, a, b = real_fourier_coefficients(func, n_modes)
        approximation = real_fourier_sum(x, a0, a, b)
        error = np.abs(approximation - target)

        if name == "Sign function":
            away = (np.abs(x) >= 0.18) & (np.abs(x) <= PI - 0.18)
        elif name == "Sawtooth  f(x)=x":
            away = np.abs(x) <= PI - 0.18
        else:
            away = np.ones_like(x, dtype=bool)

        l2_error = l2_norm(approximation - target)
        max_error = np.max(error)
        away_error = np.max(error[away])

        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        axes[0].plot(x, target, color="black", linewidth=2, label="f")
        axes[0].plot(x, approximation, color="#1565c0", label=fr"$S_{{{n_modes}}}f$")
        axes[0].legend()
        axes[0].set_title("Approximation")
        axes[1].plot(x, error, color="#c43c39")
        axes[1].set_title("Pointwise absolute error")
        axes[1].set_xlabel("x")
        plt.show()

        display(HTML(
            f"<b>L² error:</b> {l2_error:.6f} &nbsp; | &nbsp; "
            f"<b>global maximum error:</b> {max_error:.6f} &nbsp; | &nbsp; "
            f"<b>maximum away from jumps:</b> {away_error:.6f}"
        ))


convergence_button.on_click(convergence_diagnostics)
display(widgets.VBox([
    widgets.HBox([convergence_function, convergence_modes]),
    convergence_button,
    convergence_output,
]))
convergence_diagnostics()


## 6. The Dirichlet kernel

The partial sum has the convolution representation

\[
S_Nf(x)=\frac1{2\pi}\int_{-\pi}^{\pi}f(x-t)D_N(t)\,dt,
\]

where

\[
D_N(t)=\sum_{n=-N}^{N}e^{int}
=1+2\sum_{n=1}^{N}\cos(nt)
=\frac{\sin((N+\tfrac12)t)}{\sin(t/2)}.
\]

Its normalized integral is (1), but (D_N) changes sign and its (L^1)-norm grows. This lack of positivity is a central reason ordinary Fourier partial sums can behave poorly.


In [ ]:
# Explore the oscillation, normalization, and L1 size of the Dirichlet kernel.
dirichlet_order = widgets.IntSlider(value=12, min=0, max=100, step=1, description="N:")
dirichlet_button = widgets.Button(description="Draw kernel", button_style="warning", icon="wave-square")
dirichlet_output = widgets.Output()


def dirichlet_kernel(t, order):
    """Stable evaluation of D_N(t), including t=0."""
    t = np.asarray(t)
    denominator = np.sin(t / 2.0)
    numerator = np.sin((order + 0.5) * t)
    return np.where(np.abs(denominator) < 1e-12, 2 * order + 1.0, numerator / denominator)


def show_dirichlet_kernel(_=None):
    """Plot D_N and numerically verify its mass."""
    with dirichlet_output:
        clear_output(wait=True)
        order = dirichlet_order.value
        t = np.linspace(-PI, PI, 30001, endpoint=False)
        values = dirichlet_kernel(t, order)
        normalized_mass = np.mean(values)
        normalized_l1 = np.mean(np.abs(values))

        fig, ax = plt.subplots()
        ax.plot(t, values, color="#6a3d9a")
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title(fr"Dirichlet kernel $D_{{{order}}}$")
        ax.set_xlim(-PI, PI)
        plt.show()

        display(HTML(
            f"<b>(1/2π) ∫D<sub>N</sub>:</b> {normalized_mass:.9f} "
            f"&nbsp; | &nbsp; <b>(1/2π) ∫|D<sub>N</sub>|:</b> {normalized_l1:.6f}"
        ))


dirichlet_button.on_click(show_dirichlet_kernel)
display(widgets.VBox([dirichlet_order, dirichlet_button, dirichlet_output]))
show_dirichlet_kernel()


## 7. Fejér means: averaging restores positivity

The Fejér mean is the Cesàro average

\[
\sigma_Nf=\frac1{N+1}\sum_{k=0}^{N}S_kf.
\]

Equivalently,

\[
\sigma_Nf(x)=\frac1{2\pi}\int_{-\pi}^{\pi}f(x-t)K_N(t)\,dt,
\qquad
K_N(t)=\frac1{N+1}\left(\frac{\sin((N+1)t/2)}{\sin(t/2)}\right)^2\ge0.
\]

Fejér's theorem states that (sigma_Nf	o f) uniformly for every continuous periodic (f). At jumps, the means converge to the midpoint and strongly suppress the Gibbs oscillations.


In [ ]:
# Compare an ordinary partial sum with the corresponding Fejer mean.
fejer_function = widgets.Dropdown(
    options=["Sign function", "Rectangular pulse", "Absolute value  f(x)=|x|", "Smooth  f(x)=exp(cos x)"],
    value="Sign function",
    description="Function:",
)
fejer_order = widgets.IntSlider(value=30, min=1, max=150, step=1, description="N:")
fejer_button = widgets.Button(description="Compare S_N and sigma_N", button_style="primary", icon="layer-group")
fejer_output = widgets.Output()


def compare_fejer(_=None):
    """Plot and compare Fourier and Fejer approximations."""
    with fejer_output:
        clear_output(wait=True)
        name = fejer_function.value
        func = FUNCTIONS[name]
        order = fejer_order.value
        x = np.linspace(-PI, PI, 20001, endpoint=False)
        target = func(x)
        a0, a, b = real_fourier_coefficients(func, order)
        partial = real_fourier_sum(x, a0, a, b)
        averaged = fejer_sum(x, a0, a, b, order)

        if name == "Sign function":
            away = (np.abs(x) > 0.18) & (np.abs(x) < PI - 0.18)
        elif name == "Rectangular pulse":
            away = (np.abs(np.abs(x) - PI / 2) > 0.18)
        else:
            away = np.ones_like(x, dtype=bool)

        partial_error = np.max(np.abs(partial[away] - target[away]))
        fejer_error = np.max(np.abs(averaged[away] - target[away]))

        fig, ax = plt.subplots()
        ax.plot(x, target, color="black", linewidth=2.2, label="f")
        ax.plot(x, partial, color="#c43c39", alpha=0.8, label=fr"$S_{{{order}}}f$")
        ax.plot(x, averaged, color="#1565c0", linewidth=2, label=fr"$\sigma_{{{order}}}f$")
        ax.set_title("Ordinary Fourier sum versus Fejer mean")
        ax.legend(loc="best")
        plt.show()

        display(HTML(
            f"<b>Maximum error away from jumps:</b> "
            f"S<sub>N</sub> = {partial_error:.6f}, σ<sub>N</sub> = {fejer_error:.6f}."
        ))


fejer_button.on_click(compare_fejer)
display(widgets.VBox([
    widgets.HBox([fejer_function, fejer_order]),
    fejer_button,
    fejer_output,
]))
compare_fejer()


## 8. Quantifying the Gibbs phenomenon

For the sign function,

\[
\operatorname{sgn}x
\sim \frac4\pi\sum_{k=0}^{\infty}\frac{\sin((2k+1)x)}{2k+1}.
\]

Near the jump at (x=0), the oscillations become narrower as more modes are added, but the first overshoot does **not** vanish in height. Its limiting excess is

\[
G=\frac1\pi\int_0^\pi\frac{\sin t}{t}\,dt-\frac12
\approx0.0894898722
\]

times the jump size. The next experiment locates the first numerical maximum and reports the overshoot as a percentage of the jump.


In [ ]:
# Locate the Gibbs overshoot for the sign-function Fourier series.
gibbs_modes = widgets.IntSlider(value=25, min=2, max=250, step=1, description="Odd modes:")
gibbs_button = widgets.Button(description="Measure overshoot", button_style="danger", icon="chart-line")
gibbs_output = widgets.Output()


def sign_odd_partial(x, modes):
    """Use the first requested number of nonzero odd sine modes."""
    x = np.asarray(x)
    result = np.zeros_like(x, dtype=float)
    for k in range(modes):
        frequency = 2 * k + 1
        result += np.sin(frequency * x) / frequency
    return (4.0 / PI) * result


def measure_gibbs(_=None):
    """Estimate the first overshoot and compare it with the Gibbs constant."""
    with gibbs_output:
        clear_output(wait=True)
        modes = gibbs_modes.value
        x = np.linspace(0.0, min(0.8, 8.0 / modes), 30001)
        partial = sign_odd_partial(x, modes)
        index = np.argmax(partial)
        maximum = partial[index]
        location = x[index]
        jump_size = 2.0
        overshoot_percent = 100.0 * (maximum - 1.0) / jump_size
        limiting_percent = 100.0 * 0.089489872236

        fig, ax = plt.subplots()
        ax.plot(x, partial, color="#c43c39", label="Fourier partial sum")
        ax.axhline(1.0, color="black", linestyle="--", label="right-hand limit")
        ax.scatter([location], [maximum], color="#1565c0", zorder=5)
        ax.set_title("The overshoot narrows, but its relative height persists")
        ax.set_xlabel("distance to the jump")
        ax.legend()
        plt.show()

        display(HTML(
            f"<b>First maximum:</b> {maximum:.9f} at x ≈ {location:.6g}<br>"
            f"<b>Overshoot:</b> {overshoot_percent:.6f}% of the jump; "
            f"limiting value ≈ {limiting_percent:.6f}%."
        ))


gibbs_button.on_click(measure_gibbs)
display(widgets.VBox([gibbs_modes, gibbs_button, gibbs_output]))
measure_gibbs()


## 9. Smoothness controls coefficient decay

Integration by parts explains a fundamental principle:

\[
f\in C^r\text{ and periodic matching at the endpoints}
\quad\Longrightarrow\quad
\widehat f(n)=O(|n|^{-r}).
\]

A jump typically produces (O(n^{-1})) decay; a corner can produce (O(n^{-2})) decay; analytic periodic functions often have exponential decay. On a log–log graph, algebraic decay appears approximately as a straight line whose slope is the negative power.


In [ ]:
# Compare numerical coefficient decay for functions of different regularity.
decay_max_mode = widgets.IntSlider(value=60, min=15, max=150, step=5, description="Max mode:")
decay_button = widgets.Button(description="Compare decay", button_style="info", icon="arrow-down")
decay_output = widgets.Output()


def compare_decay(_=None):
    """Plot coefficient amplitudes for jump, corner, and analytic examples."""
    with decay_output:
        clear_output(wait=True)
        max_mode = decay_max_mode.value
        examples = {
            "sawtooth (jump)": FUNCTIONS["Sawtooth  f(x)=x"],
            "|x| (corner)": FUNCTIONS["Absolute value  f(x)=|x|"],
            "exp(cos x) (analytic)": FUNCTIONS["Smooth  f(x)=exp(cos x)"],
        }
        modes = np.arange(1, max_mode + 1)

        fig, ax = plt.subplots()
        summary = []
        for label, func in examples.items():
            _, a, b = real_fourier_coefficients(func, max_mode, samples=32768)
            amplitude = np.sqrt(a * a + b * b)
            visible = amplitude > 2e-13
            ax.loglog(modes[visible], amplitude[visible], marker="o", markersize=3, linewidth=1.2, label=label)

            # Estimate an algebraic slope only where the coefficients are above noise.
            usable = visible & (modes >= 5)
            if np.count_nonzero(usable) >= 4:
                slope = np.polyfit(np.log(modes[usable]), np.log(amplitude[usable]), 1)[0]
                summary.append(f"{label}: fitted log-log slope {slope:.2f}")

        ax.set_xlabel("frequency n")
        ax.set_ylabel(r"$\sqrt{a_n^2+b_n^2}$")
        ax.set_title("Regularity versus Fourier-coefficient decay")
        ax.legend()
        plt.show()
        display(HTML("<br>".join(summary) + "<br><i>Parity zeros and floating-point noise can affect a fitted slope.</i>"))


decay_button.on_click(compare_decay)
display(widgets.VBox([decay_max_mode, decay_button, decay_output]))
compare_decay()


## 10. Bessel, Parseval, and numerical series evaluation

Bessel's inequality for the real trigonometric system is

\[
\pi\left(\frac{a_0^2}{2}+\sum_{n=1}^{N}(a_n^2+b_n^2)\right)
\le \|f\|_2^2.
\]

Completeness gives Parseval's identity in the limit:

\[
\frac1\pi\int_{-\pi}^{\pi}|f(x)|^2\,dx
=\frac{a_0^2}{2}+\sum_{n=1}^{\infty}(a_n^2+b_n^2).
\]

This converts function energy into coefficient energy. It also evaluates classical series, including

\[
\sum_{n=1}^{\infty}\frac1{n^2}=\frac{\pi^2}{6},
\qquad
\sum_{n=1}^{\infty}\frac1{n^4}=\frac{\pi^4}{90},
\qquad
\sum_{k=1}^{\infty}\frac1{(2k-1)^4}=\frac{\pi^4}{96}.
\]


In [ ]:
# Visualize Parseval energy capture and a related numerical series identity.
parseval_example = widgets.Dropdown(
    options=["Sawtooth / Basel sum", "Square / zeta(4)", "Absolute value / odd fourth powers"],
    value="Sawtooth / Basel sum",
    description="Example:",
)
parseval_modes = widgets.IntSlider(value=20, min=1, max=120, step=1, description="N:")
parseval_button = widgets.Button(description="Check Parseval", button_style="success", icon="equals")
parseval_output = widgets.Output()


def check_parseval(_=None):
    """Compare coefficient energy with direct L2 energy and evaluate a series."""
    with parseval_output:
        clear_output(wait=True)
        name = parseval_example.value
        n_modes = parseval_modes.value

        if name.startswith("Sawtooth"):
            func = FUNCTIONS["Sawtooth  f(x)=x"]
            partial_series = np.sum(1.0 / np.arange(1, n_modes + 1) ** 2)
            exact_series = PI ** 2 / 6.0
            series_label = r"sum 1/n^2"
        elif name.startswith("Square"):
            func = FUNCTIONS["Square  f(x)=x^2"]
            partial_series = np.sum(1.0 / np.arange(1, n_modes + 1) ** 4)
            exact_series = PI ** 4 / 90.0
            series_label = r"sum 1/n^4"
        else:
            func = FUNCTIONS["Absolute value  f(x)=|x|"]
            odd = 2 * np.arange(1, n_modes + 1) - 1
            partial_series = np.sum(1.0 / odd ** 4)
            exact_series = PI ** 4 / 96.0
            series_label = r"sum 1/(2k-1)^4"

        x = np.linspace(-PI, PI, 40001, endpoint=False)
        values = func(x)
        a0, a, b = real_fourier_coefficients(func, n_modes, samples=40000)
        total_energy = TWO_PI * np.mean(values ** 2)
        captured_energy = PI * (a0 * a0 / 2.0 + np.sum(a * a + b * b))
        residual_energy = total_energy - captured_energy

        fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
        axes[0].bar(["captured", "remaining"], [captured_energy, max(0.0, residual_energy)], color=["#1565c0", "#cfd8dc"])
        axes[0].set_title(fr"Energy decomposition at $N={n_modes}$")
        axes[1].bar(["partial sum", "exact limit"], [partial_series, exact_series], color=["#2ca25f", "#756bb1"])
        axes[1].set_title(series_label)
        plt.show()

        display(HTML(
            f"<b>Direct energy:</b> {total_energy:.9f}<br>"
            f"<b>Captured coefficient energy:</b> {captured_energy:.9f}<br>"
            f"<b>Series error:</b> {abs(partial_series - exact_series):.3e}"
        ))


parseval_button.on_click(check_parseval)
display(widgets.VBox([
    widgets.HBox([parseval_example, parseval_modes]),
    parseval_button,
    parseval_output,
]))
check_parseval()


## 11. Half-range sine series and the heat equation

For fixed endpoints, the one-dimensional heat equation is

\[
u_t=\kappa u_{xx},
\qquad 0<x<L,
\qquad u(0,t)=u(L,t)=0.
\]

Separation of variables gives the sine-series solution

\[
u(x,t)=\sum_{n=1}^{\infty}B_n
e^{-\kappa(n\pi/L)^2t}\sin\left(\frac{n\pi x}{L}\right),
\qquad
B_n=\frac2L\int_0^L f(x)\sin\left(\frac{n\pi x}{L}\right)dx.
\]

For (f(x)=x(L-x)),

\[
B_n=\frac{4L^2(1-(-1)^n)}{n^3\pi^3}.
\]

Higher frequencies decay faster because of the factor (e^{-\kappa(n\pi/L)^2t}). Use the play button to watch diffusion smooth the initial profile.


In [ ]:
# Animate a Fourier sine-series solution of the heat equation.
heat_time = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.01, description="time t:")
heat_play = widgets.Play(value=0, min=0, max=100, step=1, interval=120, description="Play")
heat_kappa = widgets.FloatSlider(value=0.20, min=0.05, max=1.0, step=0.05, description="kappa:")
heat_modes = widgets.IntSlider(value=25, min=1, max=80, step=1, description="Modes:")
heat_output = widgets.Output()


def heat_solution(x, time, kappa, modes, length=PI):
    """Evaluate the truncated heat solution for f(x)=x(L-x)."""
    result = np.zeros_like(x, dtype=float)
    for n in range(1, modes + 1):
        coefficient = 4.0 * length ** 2 * (1.0 - (-1.0) ** n) / (n ** 3 * PI ** 3)
        decay = np.exp(-kappa * (n * PI / length) ** 2 * time)
        result += coefficient * decay * np.sin(n * PI * x / length)
    return result


def draw_heat_solution(change=None):
    """Plot the initial temperature and its diffused state."""
    with heat_output:
        clear_output(wait=True)
        x = np.linspace(0.0, PI, 1500)
        initial = x * (PI - x)
        current = heat_solution(x, heat_time.value, heat_kappa.value, heat_modes.value)

        fig, ax = plt.subplots()
        ax.plot(x, initial, color="black", linestyle="--", label="initial profile")
        ax.plot(x, current, color="#d7301f", linewidth=2.2, label=fr"$u(x,{heat_time.value:.2f})$")
        ax.fill_between(x, 0, current, color="#fc8d59", alpha=0.25)
        ax.set_ylim(0, 1.05 * np.max(initial))
        ax.set_xlabel("x")
        ax.set_ylabel("temperature")
        ax.set_title("Fourier sine modes under heat diffusion")
        ax.legend()
        plt.show()

        initial_energy = np.trapezoid(initial ** 2, x)
        current_energy = np.trapezoid(current ** 2, x)
        display(HTML(f"<b>Energy ratio ||u(·,t)||² / ||f||²:</b> {current_energy / initial_energy:.6f}"))


def advance_heat_time(change):
    """Convert the integer play position to a time in [0,1]."""
    heat_time.value = change["new"] / 100.0


heat_play.observe(advance_heat_time, names="value")
for control in (heat_time, heat_kappa, heat_modes):
    control.observe(draw_heat_solution, names="value")
display(widgets.VBox([
    widgets.HBox([heat_play, heat_time]),
    widgets.HBox([heat_kappa, heat_modes]),
    heat_output,
]))
draw_heat_solution()


## 12. Complex coefficients, translation, and convolution

The complex coefficient is

\[
\widehat f(n)=\frac1{2\pi}\int_{-\pi}^{\pi}f(x)e^{-inx}\,dx.
\]

Two structural rules are especially important:

\[
h(x)=f(x-x_0)
\quad\Longrightarrow\quad
\widehat h(n)=e^{-inx_0}\widehat f(n),
\]

and, for normalized periodic convolution,

\[
(f*g)(x)=\frac1{2\pi}\int_{-\pi}^{\pi}f(x-y)g(y)\,dy,
\qquad
\widehat{f*g}(n)=\widehat f(n)\widehat g(n).
\]

Translation changes coefficient phases but not magnitudes; convolution multiplies the coefficients mode by mode.


In [ ]:
# Verify the translation rule or the convolution theorem numerically.
complex_mode = widgets.ToggleButtons(options=["Translation", "Convolution"], description="Experiment:")
translation_shift = widgets.FloatSlider(value=0.70, min=-PI, max=PI, step=0.05, description="x0:")
complex_button = widgets.Button(description="Verify identity", button_style="primary", icon="project-diagram")
complex_output = widgets.Output()


def verify_complex_rule(_=None):
    """Run a discrete periodic experiment for a complex Fourier identity."""
    with complex_output:
        clear_output(wait=True)
        sample_count = 4096
        x = TWO_PI * np.arange(sample_count) / sample_count

        if complex_mode.value == "Translation":
            shift = translation_shift.value
            f = np.exp(np.cos(x)) + 0.35 * np.sin(3 * x)
            h = np.exp(np.cos(x - shift)) + 0.35 * np.sin(3 * (x - shift))
            f_hat = np.fft.fft(f) / sample_count
            h_hat = np.fft.fft(h) / sample_count
            frequencies = np.fft.fftfreq(sample_count, d=1.0 / sample_count)
            predicted = np.exp(-1j * frequencies * shift) * f_hat
            error = np.max(np.abs(h_hat - predicted))

            fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
            axes[0].plot(x, f, label="f(x)")
            axes[0].plot(x, h, label="f(x-x0)")
            axes[0].set_title("Translation in physical space")
            axes[0].legend()
            modes = np.arange(0, 9)
            axes[1].stem(modes, np.abs(f_hat[modes]), linefmt="#1565c0", markerfmt="o", basefmt=" ", label="|f_hat|")
            axes[1].stem(modes + 0.08, np.abs(h_hat[modes]), linefmt="#d95f02", markerfmt="s", basefmt=" ", label="|h_hat|")
            axes[1].set_title("Coefficient magnitudes are unchanged")
            axes[1].legend()
            plt.show()
            message = f"maximum coefficient identity error = {error:.3e}"
        else:
            f = np.where(x < PI, 1.0, -1.0)
            g = 1.0 + 0.60 * np.cos(x) + 0.25 * np.sin(2 * x)

            # The factor 1/M implements the normalized circular convolution.
            convolution = np.fft.ifft(np.fft.fft(f) * np.fft.fft(g)).real / sample_count
            f_hat = np.fft.fft(f) / sample_count
            g_hat = np.fft.fft(g) / sample_count
            convolution_hat = np.fft.fft(convolution) / sample_count
            predicted = f_hat * g_hat
            error = np.max(np.abs(convolution_hat - predicted))

            fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
            axes[0].plot(x, f, label="f")
            axes[0].plot(x, g, label="g")
            axes[0].legend()
            axes[0].set_title("Input functions")
            axes[1].plot(x, convolution, color="#2ca25f")
            axes[1].set_title("Normalized periodic convolution")
            plt.show()
            message = f"maximum convolution-theorem error = {error:.3e}"

        display(HTML(f"<b>Numerical certificate:</b> {message}"))


complex_button.on_click(verify_complex_rule)
display(widgets.VBox([
    complex_mode,
    translation_shift,
    complex_button,
    complex_output,
]))
verify_complex_rule()


## 13. Euler's sine product and Wallis' product

Fourier analysis of a non-integral frequency leads to the cotangent expansion

\[
\pi\cot(\pi x)=\frac1x-2x\sum_{n=1}^{\infty}\frac1{n^2-x^2}.
\]

After integrating logarithmic derivatives, one obtains Euler's product

\[
\frac{\sin(\pi x)}{\pi x}
=\prod_{n=1}^{\infty}\left(1-\frac{x^2}{n^2}\right).
\]

Setting (x=\tfrac12) gives Wallis' product

\[
\frac\pi2=\prod_{n=1}^{\infty}
\frac{2n}{2n-1}\frac{2n}{2n+1}.
\]

The experiment compares finite products with both exact limits.


In [ ]:
# Investigate convergence of Euler's sine product and Wallis' product.
product_x = widgets.FloatSlider(value=0.50, min=-2.5, max=2.5, step=0.05, description="x:")
product_terms = widgets.IntSlider(value=25, min=1, max=300, step=1, description="N:")
product_button = widgets.Button(description="Evaluate products", button_style="info", icon="times")
product_output = widgets.Output()


def evaluate_products(_=None):
    """Compare finite Euler and Wallis products with their exact targets."""
    with product_output:
        clear_output(wait=True)
        x_value = product_x.value
        terms = product_terms.value
        n = np.arange(1, terms + 1, dtype=float)
        euler_factors = 1.0 - (x_value / n) ** 2
        euler_partial = np.cumprod(euler_factors)
        euler_exact = np.sinc(x_value)
        wallis_factors = (2 * n / (2 * n - 1)) * (2 * n / (2 * n + 1))
        wallis_partial = np.cumprod(wallis_factors)

        fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
        axes[0].plot(n, euler_partial, color="#1565c0", label="finite product")
        axes[0].axhline(euler_exact, color="black", linestyle="--", label="sin(pi x)/(pi x)")
        axes[0].set_title("Euler sine product")
        axes[0].legend()
        axes[1].plot(n, wallis_partial, color="#2ca25f", label="Wallis partial product")
        axes[1].axhline(PI / 2, color="black", linestyle="--", label="pi/2")
        axes[1].set_title("Wallis product")
        axes[1].legend()
        plt.show()

        display(HTML(
            f"<b>Euler product error:</b> {abs(euler_partial[-1] - euler_exact):.3e}<br>"
            f"<b>Wallis product error:</b> {abs(wallis_partial[-1] - PI / 2):.3e}"
        ))


product_button.on_click(evaluate_products)
display(widgets.VBox([
    widgets.HBox([product_x, product_terms]),
    product_button,
    product_output,
]))
evaluate_products()


## 14. Theorem guide: choose the mathematical question first

Fourier theory contains several convergence mechanisms, and each answers a different question. A useful habit is to identify the desired conclusion before selecting a theorem:

\[
\text{geometry in }L^2,
\quad
\text{pointwise behavior},
\quad
\text{uniform approximation},
\quad
\text{energy identity},
\quad
\text{coefficient algebra}.
\]

Select a goal to see the appropriate theorem, its assumptions, and the conclusion it supports.


In [ ]:
# A theorem-selection assistant for common Fourier-analysis goals.
theorem_data = {
    "Best finite approximation": (
        "Orthogonal Projection Theorem",
        r"Use the Fourier coefficients in the chosen finite orthogonal system.",
        r"The partial sum uniquely minimizes the L2 error."
    ),
    "Pointwise limit at a jump": (
        "Dirichlet--Jordan Theorem",
        r"Assume suitable local regularity, for example piecewise C1 or bounded variation.",
        r"S_N f(x) converges to (f(x+)+f(x-))/2."
    ),
    "Uniform approximation of a continuous function": (
        "Fejer's Theorem",
        r"Assume f is continuous and periodic.",
        r"The Cesaro means sigma_N f converge uniformly to f."
    ),
    "Ordinary uniform Fourier convergence": (
        "Coefficient-decay criterion",
        r"A convenient sufficient condition is absolute summability of a_n and b_n; periodic piecewise C1 regularity plus continuity gives such decay in the chapter's setting.",
        r"The Fourier series converges absolutely and uniformly."
    ),
    "Recover total energy": (
        "Parseval's Identity",
        r"Use completeness of the trigonometric system in L2.",
        r"Function energy equals the sum of squared Fourier coordinates."
    ),
    "Analyze convolution": (
        "Fourier Convolution Theorem",
        r"Use normalized periodic convolution and complex Fourier coefficients.",
        r"The coefficient of f*g is the pointwise product of the coefficients."
    ),
}

theorem_goal = widgets.Dropdown(options=list(theorem_data), description="Goal:", layout=widgets.Layout(width="75%"))
theorem_button = widgets.Button(description="Recommend theorem", button_style="primary", icon="book-open")
theorem_output = widgets.Output()


def recommend_theorem(_=None):
    """Display the theorem, assumptions, and conclusion for the selected goal."""
    with theorem_output:
        clear_output(wait=True)
        theorem, assumptions, conclusion = theorem_data[theorem_goal.value]
        display(HTML(
            f"<div style='padding:12px;border:1px solid #aaccee;border-radius:6px'>"
            f"<h4 style='margin-top:0'>{theorem}</h4>"
            f"<b>Assumptions or mechanism:</b> {assumptions}<br>"
            f"<b>Conclusion:</b> {conclusion}"
            "</div>"
        ))


theorem_button.on_click(recommend_theorem)
display(widgets.VBox([theorem_goal, theorem_button, theorem_output]))
recommend_theorem()


## 15. Concept quiz

The quiz tests distinctions that are easy to blur:

- (int|f|^2) is the **square** of the (L^2)-norm, not the norm itself;
- Bessel's inequality does not require completeness, while Parseval does;
- (L^2) convergence does not imply pointwise convergence everywhere;
- Fourier partial sums and Fejér means have different convergence properties;
- multiplication and convolution act differently on Fourier coefficients.

Choose one answer for each question and press **Check answers**.


In [ ]:
# Self-checking conceptual quiz with immediate explanations.
quiz_questions = [
    ("Which quantity is the L2 norm?", ["integral |f|^2", "square root of integral |f|^2", "integral |f|"], 1,
     "The L2 norm is the square root of the energy integral."),
    ("What does Bessel's inequality require?", ["An orthonormal system", "A complete system", "Pointwise convergence"], 0,
     "Orthogonality is sufficient; completeness is needed to obtain equality for every f."),
    ("At a jump, a piecewise smooth Fourier series converges to:", ["the left limit", "the right limit", "the midpoint of the one-sided limits"], 2,
     "This is the Dirichlet--Jordan midpoint rule."),
    ("For every continuous periodic f, which approximation converges uniformly?", ["Fourier partial sums", "Fejer means", "Both without further assumptions"], 1,
     "Fejer's positivity theorem applies to every continuous periodic function."),
    ("The Gibbs overshoot as N grows:", ["vanishes in relative height", "persists in relative height but narrows", "grows without bound"], 1,
     "Its width shrinks, while its limiting excess is about 8.949% of the jump."),
    ("Translation f(x-x0) does what to the nth complex coefficient?", ["multiplies it by exp(-i n x0)", "changes only its magnitude", "replaces n by n-x0"], 0,
     "Translation produces a phase factor and leaves coefficient magnitude unchanged."),
    ("Periodic convolution becomes:", ["discrete convolution of coefficients", "pointwise multiplication of coefficients", "differentiation of coefficients"], 1,
     "Normalized periodic convolution becomes coefficientwise multiplication."),
]

quiz_controls = []
for number, (question, options, _, _) in enumerate(quiz_questions, start=1):
    control = widgets.RadioButtons(options=options, description=f"{number}.", layout=widgets.Layout(width="95%"))
    quiz_controls.append(widgets.VBox([widgets.HTML(f"<b>{question}</b>"), control]))

quiz_button = widgets.Button(description="Check answers", button_style="success", icon="check-circle")
quiz_output = widgets.Output()


def grade_quiz(_=None):
    """Grade all answers and explain any incorrect choice."""
    with quiz_output:
        clear_output(wait=True)
        score = 0
        feedback = []
        for number, ((question, options, correct, explanation), box) in enumerate(zip(quiz_questions, quiz_controls), start=1):
            selected = box.children[1].value
            if selected == options[correct]:
                score += 1
                feedback.append(f"<span style='color:#198754'>✓ {number}. Correct.</span>")
            else:
                feedback.append(f"<span style='color:#b02a37'>✗ {number}. {explanation}</span>")
        display(HTML(f"<h4>Score: {score}/{len(quiz_questions)}</h4>" + "<br>".join(feedback)))


quiz_button.on_click(grade_quiz)
display(widgets.VBox(quiz_controls + [quiz_button, quiz_output]))


## 16. Practice generator with guided solutions

Active practice is most effective when computation is paired with a theorem. The generator draws from four levels:

1. coefficient and parity calculations;
2. convergence classification;
3. Parseval and numerical series;
4. structural rules and PDE applications.

Try to write a complete solution before revealing the guide. In particular, state the relevant hypotheses and the precise mode of convergence.


In [ ]:
# Generate a practice problem and reveal a concise solution on demand.
practice_bank = {
    "Foundations": [
        (r"Show that the sine coefficients of every even integrable function vanish.",
         r"The product f(x) sin(nx) is odd. Its integral over [-pi,pi] is therefore zero, so b_n=0."),
        (r"Find the Fourier sine coefficients of f(x)=x on (-pi,pi).",
         r"Oddness gives a_n=0. Integration by parts gives b_n=2(-1)^(n+1)/n."),
    ],
    "Convergence": [
        (r"Determine the Fourier-series limit of the periodic sawtooth at x=pi.",
         r"The one-sided limits are pi and -pi. Dirichlet--Jordan gives their midpoint, 0."),
        (r"Explain why the sign-function Fourier series cannot converge uniformly on a full period.",
         r"Every partial sum is continuous, but its normalized pointwise limit has jumps. A uniform limit of continuous functions is continuous."),
    ],
    "Parseval": [
        (r"Use the sawtooth coefficients to derive the Basel sum.",
         r"For f(x)=x, b_n=2(-1)^(n+1)/n. Parseval gives 2 pi^2/3 = pi times 4 sum 1/n^2, hence sum 1/n^2=pi^2/6."),
        (r"For f(x)=|x|, identify the series obtained from Parseval.",
         r"Use a_0=pi and a_(2k-1)=-4/[pi(2k-1)^2]. Parseval yields sum 1/(2k-1)^4=pi^4/96."),
    ],
    "Applications": [
        (r"If h(x)=f(x-x0), derive the formula for h-hat(n).",
         r"Substitute y=x-x0 and use periodicity. The exponential contributes exp(-i n x0), so h-hat(n)=exp(-i n x0) f-hat(n)."),
        (r"Why do high-frequency sine modes disappear faster in the heat equation?",
         r"Mode n is multiplied by exp[-kappa(n pi/L)^2 t]. The exponent is quadratic in n, so high frequencies decay much faster."),
    ],
}

practice_level = widgets.Dropdown(options=list(practice_bank), description="Level:")
new_problem_button = widgets.Button(description="New problem", button_style="primary", icon="random")
solution_button = widgets.Button(description="Reveal solution", button_style="warning", icon="lightbulb")
practice_problem = widgets.HTMLMath()
practice_solution = widgets.HTMLMath()
practice_state = {"solution": ""}
practice_rng = np.random.default_rng(20260717)


def new_problem(_=None):
    """Choose a reproducible random problem from the selected level."""
    candidates = practice_bank[practice_level.value]
    index = int(practice_rng.integers(0, len(candidates)))
    problem, solution = candidates[index]
    practice_state["solution"] = solution
    practice_problem.value = f"<div style='padding:12px;background:#eef6ff'><b>Problem.</b> {problem}</div>"
    practice_solution.value = ""


def reveal_solution(_=None):
    """Reveal the stored guide for the current problem."""
    practice_solution.value = (
        "<div style='padding:12px;background:#fff4d6;border-left:4px solid #e0a800'>"
        f"<b>Solution guide.</b> {practice_state['solution']}</div>"
    )


new_problem_button.on_click(new_problem)
solution_button.on_click(reveal_solution)
practice_level.observe(new_problem, names="value")
display(widgets.VBox([
    widgets.HBox([practice_level, new_problem_button, solution_button]),
    practice_problem,
    practice_solution,
]))
new_problem()


## Suggested learning path

1. Begin with the projection experiment and explain why the Fourier coefficients are forced by orthogonality.
2. Compare a continuous function, a function with a corner, and a function with a jump.
3. Use the Dirichlet and Fejér cells together: relate kernel sign, averaging, and convergence.
4. Measure Gibbs overshoot for increasingly many modes.
5. Verify Parseval before using it to evaluate a numerical series.
6. Finish with the heat equation and the complex coefficient rules.

### Final checkpoint

You should now be able to distinguish the following statements without ambiguity:

\[
f\sim\sum c_n\phi_n,
\qquad
S_Nf\xrightarrow{L^2}f,
\qquad
S_Nf(x)\to f(x),
\qquad
\|S_Nf-f\|_\infty\to0.
\]

The symbol (sim) records the coefficients; it does **not** by itself assert any of the three convergence statements.
